In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28,28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)

    transforms.ToTensor(),
    # TODO: Convert to Tensor
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
       transforms.Normalize(mean=[0.485, 0.456, 0.406], # These random ahh values were found using ImageNet dataset, https://paperswithcode.com/dataset/imagenet
                          std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

import matplotlib.pyplot as plt
import numpy as np

# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].set_title(f'Class: {label}')
    axes[i].axis('off')

plt.show()


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import EMNIST
from PIL import Image # Import PIL Image for custom transform

# 1. Custom Transform: Convert 1-channel PIL image to 3-channel PIL image
class GrayscaleToRGB:
    """Converts a grayscale PIL Image to an RGB PIL Image."""
    def __call__(self, img):
        if img.mode == 'L': # 'L' mode indicates grayscale
            return img.convert('RGB')
        return img

# 2. Custom Transform: Adjust EMNIST 'letters' labels from 1-indexed to 0-indexed
class SubtractOne:
    """Subtracts 1 from the label to convert 1-indexed to 0-indexed."""
    def __call__(self, label):
        return label - 1

class CustomModel(nn.Module):
    def __init__(self, num_classes=26, efficientnet_input_size=(28,28)):
        super(CustomModel, self).__init__()

        # Load pre-trained EfficientNetV2_S model
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

        # Freeze the feature extractor (backbone) layers
        for param in model.features.parameters():
            param.requires_grad = False

        self.features = model.features

        # AdaptiveAvgPool2d will reduce the spatial dimensions of the feature map to (1, 1)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # --- Dynamic calculation of num_features ---
        # The dummy input MUST be 3 channels as the EfficientNet expects it.
        # This is consistent with the GrayscaleToRGB transform now.
        dummy_input = torch.randn(1, 3, efficientnet_input_size[0], efficientnet_input_size[1])

        with torch.no_grad():
            # Temporarily set the feature extractor to evaluation mode for this dummy pass.
            # This is crucial to prevent BatchNorm layers from trying to compute statistics
            # on a batch of 1 when spatial dimensions might have collapsed to 1x1.
            self.features.eval()
            output_features = self.features(dummy_input)
            output_features = self.avgpool(output_features)
            # Restore to train mode. For frozen features, their behavior will be like eval mode anyway.
            self.features.train()

        num_features = torch.flatten(output_features, 1).size(1)
        print(f"Number of features from EfficientNetV2_S for {efficientnet_input_size[0]}x{efficientnet_input_size[1]} input: {num_features}")
        # As noted before, for 28x28 input, the output features are still 1280 (after avgpool)
        # for EfficientNetV2_S, as internal spatial dimensions collapse early.

        # Custom classifier head for EMNIST letters
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(True),
            nn.Dropout(0.5), # Increased dropout for robustness
            nn.Linear(512, 256),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes) # Final layer with num_classes outputs
        )

    def forward(self, x):
        # EMNIST images are now already 3 channels and normalized due to the transforms.Compose
        # The line `x = x.repeat(1, 3, 1, 1)` is no longer needed here.

        # Pass through the frozen feature extractor
        x = self.features(x)

        # Apply global average pooling to get a (batch_size, num_features, 1, 1) tensor
        x = self.avgpool(x)

        # Flatten the output for the classifier: (batch_size, num_features)
        x = torch.flatten(x, 1)

        # Pass through the custom classifier
        x = self.classifier(x)
        return x



In [ ]:
# Write your code here



In [ ]:
# Initialize the model AFTER data checks to ensure print statements are in order
model = CustomModel(num_classes=26)
print(model)


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validate_epoch(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.classifier.parameters(), lr=0.001, momentum=0.9)

num_epochs = 5

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model,train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print("Finished Training.")

In [ ]:
# Write your code here
